In [ ]:
import pandas as pd
import numpy as np
df=pd.read_csv("diabetes_prediction_dataset.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#Check null values
df.isnull().sum()

In [ ]:
#Check duplicate rows
print("Duplicate Rows:",df.duplicated().sum())

In [ ]:
#Data cleaning
#Remove duplicate rows and gender "Other" (only 18 rows)
df=df.drop_duplicates()
df=df[df["gender"]!="Other"]
df=df.reset_index(drop=True)
print("Shape after cleaning:",df.shape)

In [ ]:
#EDA - class balance
import matplotlib.pyplot as plt
import seaborn as sns
print(df["diabetes"].value_counts())
print(df["diabetes"].value_counts(normalize=True))
sns.countplot(x="diabetes",data=df)
plt.title("Class balance")
plt.show()

In [ ]:
#Numeric features vs diabetes
fig,axes=plt.subplots(1,4,figsize=(18,4))
for ax,col in zip(axes,["age","bmi","HbA1c_level","blood_glucose_level"]):
    sns.boxplot(x="diabetes",y=col,data=df,ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
#Diabetes rate in each category
for col in ["gender","smoking_history","hypertension","heart_disease"]:
    print(df.groupby(col)["diabetes"].mean().round(3))
    print()

In [ ]:
#Correlation
plt.figure(figsize=(8,6))
sns.heatmap(df[["age","hypertension","heart_disease","bmi","HbA1c_level","blood_glucose_level","diabetes"]].corr(),annot=True,fmt=".2f",cmap="coolwarm")
plt.show()

In [ ]:
#Important pattern in this dataset
print("HbA1c > 6.6 :",df[df["HbA1c_level"]>6.6]["diabetes"].mean())
print("Glucose > 200 :",df[df["blood_glucose_level"]>200]["diabetes"].mean())
print("HbA1c <= 5.0 :",df[df["HbA1c_level"]<=5.0]["diabetes"].mean())
print("Glucose <= 100 :",df[df["blood_glucose_level"]<=100]["diabetes"].mean())

In [ ]:
#Convert text columns (gender, smoking_history) to numbers
data=pd.get_dummies(df,columns=["gender","smoking_history"],drop_first=True)
X=data.drop(columns=["diabetes"]).astype(float)
y=data["diabetes"]
X.head()

In [ ]:
#Splitting the dataset
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)
print(f"Training features:{X_train.shape},Training labels:{y_train.shape}")
print(f"Testing features: {X_test.shape},Testing labels:{y_test.shape}")

In [ ]:
#Scaling (needed for KNN and Logistic Regression)
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)

In [ ]:
#Baseline: accuracy if we always predict "No diabetes"
print("Baseline accuracy:",1-y_test.mean())

In [ ]:
#KNN
from sklearn.neighbors import KNeighborsClassifier
knn=KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s,y_train)
y_pred_knn=knn.predict(X_test_s)

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score

#Precision, recall and F1 are for the diabetic class (1)
print("Accuracy:",accuracy_score(y_test,y_pred_knn))
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_knn,labels=[0,1]))
print("Precision:",precision_score(y_test,y_pred_knn))
print("Recall:",recall_score(y_test,y_pred_knn))
print("F1 Score:",f1_score(y_test,y_pred_knn))

In [ ]:
#Logistic regression
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression(max_iter=2000)
lr.fit(X_train_s,y_train)
y_pred_lr=lr.predict(X_test_s)

In [ ]:
print("Accuracy:",accuracy_score(y_test,y_pred_lr))
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_lr,labels=[0,1]))
print("Precision:",precision_score(y_test,y_pred_lr))
print("Recall:",recall_score(y_test,y_pred_lr))
print("F1 Score:",f1_score(y_test,y_pred_lr))

In [ ]:
#Naive Bayes (uses Bayes theorem, scaling not needed)
from sklearn.naive_bayes import GaussianNB
Model_NB=GaussianNB()
Model_NB.fit(X_train,y_train)
y_pred_nb=Model_NB.predict(X_test)

In [ ]:
print("Accuracy:",accuracy_score(y_test,y_pred_nb))
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_nb,labels=[0,1]))
print("Precision:",precision_score(y_test,y_pred_nb))
print("Recall:",recall_score(y_test,y_pred_nb))
print("F1 Score:",f1_score(y_test,y_pred_nb))

In [ ]:
#Only about 9% of patients are diabetic, so the models miss many of them (low recall)
#Balance the TRAINING data (2 non-diabetic rows for every 1 diabetic row), test data is not changed
pos=y_train[y_train==1].index
neg=np.random.RandomState(42).choice(y_train[y_train==0].index,size=2*len(pos),replace=False)
bal_idx=pos.union(pd.Index(neg))
X_bal=X_train.loc[bal_idx]
y_bal=y_train.loc[bal_idx]
X_bal_s=scaler.transform(X_bal)
print("Balanced training data shape:",X_bal.shape)
print(y_bal.value_counts())

In [ ]:
#KNN on balanced data
knn2=KNeighborsClassifier(n_neighbors=5)
knn2.fit(X_bal_s,y_bal)
y_pred_knn2=knn2.predict(X_test_s)

print("Accuracy:",accuracy_score(y_test,y_pred_knn2))
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_knn2,labels=[0,1]))
print("Precision:",precision_score(y_test,y_pred_knn2))
print("Recall:",recall_score(y_test,y_pred_knn2))
print("F1 Score:",f1_score(y_test,y_pred_knn2))

In [ ]:
#Logistic regression on balanced data
lr2=LogisticRegression(max_iter=2000)
lr2.fit(X_bal_s,y_bal)
y_pred_lr2=lr2.predict(X_test_s)

print("Accuracy:",accuracy_score(y_test,y_pred_lr2))
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_lr2,labels=[0,1]))
print("Precision:",precision_score(y_test,y_pred_lr2))
print("Recall:",recall_score(y_test,y_pred_lr2))
print("F1 Score:",f1_score(y_test,y_pred_lr2))

In [ ]:
#Naive Bayes on balanced data
nb2=GaussianNB()
nb2.fit(X_bal,y_bal)
y_pred_nb2=nb2.predict(X_test)

print("Accuracy:",accuracy_score(y_test,y_pred_nb2))
print("Confusion Matrix:")
print(confusion_matrix(y_test,y_pred_nb2,labels=[0,1]))
print("Precision:",precision_score(y_test,y_pred_nb2))
print("Recall:",recall_score(y_test,y_pred_nb2))
print("F1 Score:",f1_score(y_test,y_pred_nb2))

In [ ]:
#Comparison of all models
predictions={"KNN":y_pred_knn,"Logistic Regression":y_pred_lr,"Naive Bayes":y_pred_nb,
             "KNN (balanced)":y_pred_knn2,"Logistic Regression (balanced)":y_pred_lr2,"Naive Bayes (balanced)":y_pred_nb2}
rows=[]
for name,pred in predictions.items():
    rows.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred),recall_score(y_test,pred),f1_score(y_test,pred)])
pd.DataFrame(rows,columns=["Model","Accuracy","Precision","Recall","F1 Score"]).round(3)

In [ ]:
#Prediction for a new patient (using the models trained on balanced data)
age=float(input("Enter the age: "))
hypertension=int(input("Enter hypertension (Yes=1,No=0): "))
heart_disease=int(input("Enter heart disease (Yes=1,No=0): "))
bmi=float(input("Enter the BMI: "))
HbA1c=float(input("Enter the HbA1c level: "))
glucose=int(input("Enter the blood glucose level: "))
gender=input("Enter the gender (Male/Female): ")
smoking=input("Enter smoking history (never/former/current/not current/ever/No Info): ")
new_patient=pd.DataFrame([[gender,age,hypertension,heart_disease,smoking,bmi,HbA1c,glucose]],
                         columns=["gender","age","hypertension","heart_disease","smoking_history","bmi","HbA1c_level","blood_glucose_level"])
#Same encoding as the training data
new_patient=pd.get_dummies(new_patient,columns=["gender","smoking_history"]).reindex(columns=X.columns,fill_value=0).astype(float)
new_scaled=scaler.transform(new_patient)
result={0:"Not Diabetic",1:"Diabetic"}
print("KNN:",result[knn2.predict(new_scaled)[0]])
print("Logistic Regression:",result[lr2.predict(new_scaled)[0]])
print("Naive Bayes:",result[nb2.predict(new_patient)[0]])

In [ ]:
#NB Decision Boundary (HbA1c and glucose)
from sklearn.inspection import DecisionBoundaryDisplay
X_data=X_train[["HbA1c_level","blood_glucose_level"]]
Model=GaussianNB()
Model.fit(X_data,y_train)
DecisionBoundaryDisplay.from_estimator(Model,X_data,response_method="predict",alpha=0.3,cmap="Set1")
sample=X_data.sample(2000,random_state=42)
plt.scatter(sample["HbA1c_level"],sample["blood_glucose_level"],c=y_train.loc[sample.index],cmap="Set1",s=8)
plt.xlabel("HbA1c level")
plt.ylabel("Blood glucose level")
plt.title("NB Decision Boundary")
plt.show()